In [2]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.config import load_config
from src.models.embeddings import encode_news

cfg = load_config("small")
OUT = cfg["paths"]["processed"]

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TAG = "minilm"                                   # sufijo para los archivos y resultados
EMB_PATH = OUT / f"news_emb_{TAG}"

news_pdf = pd.read_parquet(OUT / "news", columns=["news_id", "text", "title"])
print("Noticias a codificar:", len(news_pdf))

t0 = time.time()
emb = encode_news(news_pdf, MODEL_NAME)
print(f"Embeddings: {emb.shape} en {time.time() - t0:.0f} s")

pd.DataFrame({"news_id": news_pdf["news_id"], "emb": list(emb.astype(np.float32))}) \
  .to_parquet(EMB_PATH, index=False)
print("Guardado en", EMB_PATH)

Noticias a codificar: 65238


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/510 [00:00<?, ?it/s]

Embeddings: (65238, 384) en 864 s
Guardado en C:\DevProjects\MMDS\RecSysMIND\data\processed\small\news_emb_minilm


In [3]:
def vecinos(texto_busqueda, n=6):
    """Noticias más similares (por embedding) a la primera que contenga el texto."""
    i = news_pdf.index[news_pdf["title"].str.contains(texto_busqueda, case=False, na=False)][0]
    sims = emb @ emb[i]
    top = np.argsort(-sims)[:n]
    print(f"Consulta: {news_pdf.loc[i, 'title']}\n")
    for j in top[1:]:
        print(f"  {sims[j]:.3f}  {news_pdf.loc[j, 'title']}")
vecinos("Olajuwon")

Consulta: Hakeem Olajuwon, Clyde Drexler share a moment before Astros' World Series Game 6

  0.750  One win from title, Astros enlist Olajuwon, Drexler for first pitch
  0.549  Astros finally get timely hitting to get back into World Series
  0.529  Russell Westbrook, James Harden cheer on the Astros behind the plate in Game 6
  0.526  Both spurned by Paul George, Thunder and Pacers face off with strong replacements
  0.521  Thunder: Both spurned by Paul George, Thunder and Pacers face off with strong replacements


In [4]:
from pyspark.sql import functions as F
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.embeddings import user_profiles_dense, score_dense

spark = get_spark(cfg)
RES = ROOT / "results" / "metrics"
HIST_KS = (None, 30, 10)

imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))
news_emb = spark.read.parquet(str(EMB_PATH)).cache()

users_val = (beh.filter("periodo = 'val'")
                .select("user_id", "history", "hist_len")
                .dropDuplicates(["user_id"]).cache())

p0 = imp.filter("periodo = 'train'").agg(F.avg("label")).first()[0]
base = (imp.filter(F.col("periodo").isin("train", "val"))
           .filter(F.col("time") >= F.to_timestamp(F.lit("2019-11-13 00:00:00")))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))

cands = (add_recent_counts(base, hours=(24,))
         .filter("periodo = 'val'")
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .join(F.broadcast(users_val.select("user_id", "hist_len")), "user_id")
         .withColumn("con_hist", F.col("hist_len") > 0)
         .select("impression_id", "user_id", "news_id", "label", "ctr", "con_hist")
         .cache())
print("Candidatas de val:", cands.count())

Candidatas de val: 1222429


In [5]:
def eval_score(df, score_col, nombre, conjunto):
    per_imp, res = evaluate(df.withColumn("score", F.col(score_col)))
    per_imp.unpersist()
    return {"modelo": nombre, "conjunto": conjunto, **res}

filas = []
for k in HIST_KS:
    t0 = time.time()
    hist_txt = "completo" if k is None else f"ultimas{k}"

    profiles = user_profiles_dense(users_val, news_emb, k=k).cache()
    scored = (score_dense(cands, news_emb, profiles)
              .withColumn("hibrido", F.when(F.col("con_hist"), F.col("cos")).otherwise(F.col("ctr"))))

    filas.append(eval_score(scored.filter("con_hist"), "cos", f"emb_{TAG}_{hist_txt}", "con_historial"))
    filas.append(eval_score(scored, "hibrido", f"hibrido_emb_{TAG}_{hist_txt}", "todas"))

    print(f"{hist_txt:9s} AUC con_hist = {filas[-2]['auc']:.4f} | "
          f"AUC híbrido (todas) = {filas[-1]['auc']:.4f} ({time.time() - t0:.0f} s)")
    profiles.unpersist()

completo  AUC con_hist = 0.6387 | AUC híbrido (todas) = 0.6386 (90 s)
ultimas30 AUC con_hist = 0.6362 | AUC híbrido (todas) = 0.6361 (78 s)
ultimas10 AUC con_hist = 0.6242 | AUC híbrido (todas) = 0.6244 (84 s)


In [6]:
tabla = (pd.DataFrame(filas)[["modelo", "conjunto", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
         .sort_values(["conjunto", "auc"], ascending=[True, False]).reset_index(drop=True))
(tabla.assign(periodo="val", dataset=cfg["dataset"])
      .to_csv(RES / f"04b_contenido_emb_{TAG}_val.csv", index=False))
tabla.round(4)

,modelo,conjunto,auc,mrr,ndcg5,ndcg10,impresiones
0,emb_minilm_completo,con_historial,0.6387,0.2949,0.3171,0.3761,29498
1,emb_minilm_ultimas30,con_historial,0.6362,0.2934,0.3150,0.3741,29498
2,emb_minilm_ultimas10,con_historial,0.6242,0.2851,0.3056,0.3643,29498
3,hibrido_emb_minilm_completo,todas,0.6386,0.2942,0.3161,0.3752,30270
4,hibrido_emb_minilm_ultimas30,todas,0.6361,0.2928,0.3140,0.3733,30270
5,hibrido_emb_minilm_ultimas10,todas,0.6244,0.2847,0.3049,0.3638,30270


In [ ]:
spark.stop() 